# Chaos and Failure Injection

**Phase 07** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-07/12-chaos-failure-injection.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You have a circuit breaker in your LLM service. It is configured to open after 5 consecutive errors and fall back to a cached response. You are confident it works. You have read the code.

You have never run it.

At 2 a.m. the Anthropic API has a partial outage. Requests start returning 529 (service overloaded). Your circuit breaker is supposed to open and serve cached responses. Instead, your service throws an unhandled exception on the first 529. The circuit breaker never fires because it only handles HTTP 5xx as an error, not 529 specifically. All users see a 500 error for 40 minutes.

This is the test gap chaos engineering fills. Fallback code that is never executed in a test is untested...

## The Concept

### The 5 LLM-Specific Failure Modes



| Failure | HTTP Status | What breaks | Correct behavior |
|---------|-------------|-------------|-----------------|
| Timeout | (connection times out) | Hangs indefinitely | Raise after timeout_seconds; retry once |
| Rate limit | 429 | Hammers API on retry | Backoff using retry-after header; max 3 retries |
| Malformed JSON | 200 | JSON parser throws | Catch parse error; return error response to user |
| Empty response | 200 | Downstream expects content | Check output_tokens > 0; retry once |
| Overload | 529 | Often unhandled | Treat as temporary; exp...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Chaos Test Suite - Phase 07, Lesson 12
Injects the 5 LLM-specific failure modes and verifies service recovery.

Usage:
    # Mock mode (no API key needed)
    python main.py

    # Real Anthropic client
    ANTHROPIC_API_KEY=sk-ant-... python main.py --real-client
"""

from __future__ import annotations

import argparse
import asyncio
import dataclasses
import json
import time
from enum import Enum
from typing import Any, Optional
from unittest.mock import MagicMock, patch


class FailureMode(str, Enum):
    TIMEOUT = "timeout"
    RATE_LIMIT = "rate_limit"
    MALFORMED_JSON = "malformed_json"
    EMPTY_RESPONSE = "empty_response"
    OVERLOAD_529 = "overload_529"


@dataclasses.dataclass
class LLMResponse:
    status: str           # "ok", "error", "fallback"
    content: str
    retry_count: int = 0
    error_type: Optional[str] = None
    duration_ms: float = 0.0


class ChaosError(Exception):
    """Base class for injected failures."""
    def __init__(self, status_code: int, message: str, retry_after: int = 0):
        self.status_code = status_code
        self.retry_after = retry_after
        super().__init__(message)


class ChaosProxy:
    """
    Wraps an LLM client and injects failures on demand.

    Args:
        failure_mode: Which failure to inject.
        failure_rate: Fraction of calls that should fail (0.0 to 1.0).
            1.0 means every call fails. 0.5 means every other call fails.
        max_failures: Stop injecting after this many failures (simulates transient issues).
    """

    def __init__(
        self,
        failure_mode: FailureMode,
        failure_rate: float = 1.0,
        max_failures: int = 100,
    ):
        self.failure_mode = failure_mode
        self.failure_rate = failure_rate
        self.max_failures = max_failures
        self._failure_count = 0
        self._call_count = 0

    def _should_fail(self) -> bool:
        self._call_count += 1
        if self._failure_count >= self.max_failures:
            return False
        import random
        return random.random() < self.failure_rate

    def messages_create(self, **kwargs) -> Any:
        """Mimics anthropic.Anthropic().messages.create() with injected failures."""
        if not self._should_fail():
            return self._ok_response(kwargs.get("max_tokens", 100))

        self._failure_count += 1

        if self.failure_mode == FailureMode.TIMEOUT:
            # Simulate a connection that never responds
            time.sleep(0.1)  # Shortened for test speed; real timeout would be 30s
            raise TimeoutError("Connection timed out after 30 seconds")

        elif self.failure_mode == FailureMode.RATE_LIMIT:
            error = ChaosError(
                status_code=429,
                message="Rate limit exceeded. Too many tokens per minute.",
                retry_after=2,
            )
            raise error

        elif self.failure_mode == FailureMode.MALFORMED_JSON:
            # Return an object that will fail when accessed like a real response
            bad_response = MagicMock()
            bad_response.content = None
            bad_response.model_dump_json.side_effect = json.JSONDecodeError(
                "Expecting value", "", 0
            )
            # Simulate the proxy returning a corrupted response object
            raise ValueError("Response body is not valid JSON: Expecting value: line 1 column 1")

        elif self.failure_mode == FailureMode.EMPTY_RESPONSE:
            # Valid response structure but zero output tokens
            response = MagicMock()
            response.content = []
            response.stop_reason = "stop"
            response.usage.output_tokens = 0
            response.usage.input_tokens = 50
            return response

        elif self.failure_mode == FailureMode.OVERLOAD_529:
            raise ChaosError(
                status_code=529,
                message="Overloaded. Please retry after a brief wait.",
                retry_after=5,
            )

        return self._ok_response(kwargs.get("max_tokens", 100))

    def _ok_response(self, max_tokens: int) -> MagicMock:
        response = MagicMock()
        response.content = [MagicMock()]
        response.content[0].text = "This is a valid response from the mock."
        response.stop_reason = "end_turn"
        response.usage.output_tokens = min(50, max_tokens)
        response.usage.input_tokens = 30
        return response


class LLMServiceUnderTest:
    """
    A minimal LLM service with production-grade error handling.
    This is what you are testing. It should handle all 5 failure modes gracefully.
    """

    MAX_RETRIES = 3
    TIMEOUT_SECONDS = 5.0
    CIRCUIT_BREAKER_THRESHOLD = 5

    def __init__(self, client: Optional[ChaosProxy] = None):
        self.client = client
        self._consecutive_failures = 0
        self._circuit_open = False

    def answer_question(self, question: str) -> LLMResponse:
        """Call the LLM to answer a question, with full error handling."""
        start = time.perf_counter()

        if self._circuit_open:
            return LLMResponse(
                status="fallback",
                content="Service temporarily unavailable. Please try again later.",
                error_type="circuit_open",
                duration_ms=(time.perf_counter() - start) * 1000,
            )

        last_error = None
        for attempt in range(self.MAX_RETRIES):
            try:
                response = self.client.messages_create(
                    model="claude-3-5-haiku-20241022",
                    messages=[{"role": "user", "content": question}],
                    max_tokens=200,
                )

                # Handle empty response
                if not response.content or response.usage.output_tokens == 0:
                    if attempt < self.MAX_RETRIES - 1:
                        time.sleep(0.1)
                        continue
                    return LLMResponse(
                        status="error",
                        content="Received empty response from model.",
                        retry_count=attempt + 1,
                        error_type="empty_response",
                        duration_ms=(time.perf_counter() - start) * 1000,
                    )

                # Success
                self._consecutive_failures = 0
                return LLMResponse(
                    status="ok",
                    content=response.content[0].text,
                    retry_count=attempt,
                    duration_ms=(time.perf_counter() - start) * 1000,
                )

            except TimeoutError as e:
                last_error = e
                self._consecutive_failures += 1
                if attempt < self.MAX_RETRIES - 1:
                    time.sleep(0.5 * (attempt + 1))  # linear backoff for timeouts
                continue

            except ChaosError as e:
                last_error = e
                self._consecutive_failures += 1

                if e.status_code == 429:
                    # Respect retry-after header
                    retry_wait = min(e.retry_after, 2)  # cap for tests
                    if attempt < self.MAX_RETRIES - 1:
                        time.sleep(retry_wait)
                    continue

                if e.status_code == 529:
                    # Exponential backoff for overload
                    wait = min(2 ** attempt * 0.5, 4)
                    if attempt < self.MAX_RETRIES - 1:
                        time.sleep(wait)
                    # Check circuit breaker
                    if self._consecutive_failures >= self.CIRCUIT_BREAKER_THRESHOLD:
                        self._circuit_open = True
                    continue

                # Unknown status code
                return LLMResponse(
                    status="error",
                    content=f"API error {e.status_code}: {str(e)}",
                    retry_count=attempt + 1,
                    error_type=f"api_{e.status_code}",
                    duration_ms=(time.perf_counter() - start) * 1000,
                )

            except (ValueError, json.JSONDecodeError) as e:
                # Malformed response - do not retry
                return LLMResponse(
                    status="error",
                    content="Received malformed response from API.",
                    retry_count=attempt,
                    error_type="malformed_response",
                    duration_ms=(time.perf_counter() - start) * 1000,
                )

        # All retries exhausted
        return LLMResponse(
            status="error",
            content="Service unavailable after retries.",
            retry_count=self.MAX_RETRIES,
            error_type=type(last_error).__name__ if last_error else "unknown",
            duration_ms=(time.perf_counter() - start) * 1000,
        )


@dataclasses.dataclass
class ChaosTestResult:
    mode: FailureMode
    passed: bool
    message: str
    duration_ms: float


def run_chaos_suite() -> list[ChaosTestResult]:
    results = []

    test_cases = [
        (
            FailureMode.TIMEOUT,
            lambda r: r.status in ("error", "fallback") and r.retry_count > 0,
            "Service must return non-ok status and show retry attempts",
        ),
        (
            FailureMode.RATE_LIMIT,
            lambda r: r.status in ("error", "fallback") and r.retry_count > 0,
            "Service must back off and retry on 429",
        ),
        (
            FailureMode.MALFORMED_JSON,
            lambda r: r.status == "error" and r.error_type == "malformed_response",
            "Service must catch parse error and return structured error",
        ),
        (
            FailureMode.EMPTY_RESPONSE,
            lambda r: r.status in ("error", "ok") and r.retry_count >= 1,
            "Service must detect empty response and retry at least once",
        ),
        (
            FailureMode.OVERLOAD_529,
            lambda r: r.status in ("error", "fallback"),
            "Service must handle 529 with backoff and not crash",
        ),
    ]

    for failure_mode, assertion_fn, description in test_cases:
        proxy = ChaosProxy(failure_mode=failure_mode, failure_rate=1.0)
        service = LLMServiceUnderTest(client=proxy)

        start = time.perf_counter()
        try:
            result = service.answer_question("What is the capital of France?")
            duration_ms = (time.perf_counter() - start) * 1000
            passed = assertion_fn(result)
            message = (
                f"Service returned status='{result.status}', retry_count={result.retry_count}."
                if passed else
                f"FAIL: {description}. Got status='{result.status}', error_type='{result.error_type}'"
            )
        except Exception as e:
            duration_ms = (time.perf_counter() - start) * 1000
            passed = False
            message = f"FAIL: Unhandled exception: {type(e).__name__}: {str(e)[:100]}"

        results.append(ChaosTestResult(
            mode=failure_mode,
            passed=passed,
            message=message,
            duration_ms=duration_ms,
        ))

    return results


def main():
    parser = argparse.ArgumentParser(description="LLM chaos test suite")
    parser.add_argument("--real-client", action="store_true",
                        help="Use real Anthropic client (requires ANTHROPIC_API_KEY)")
    args = parser.parse_args()

    print("Running chaos test suite...\n")
    results = run_chaos_suite()

    passed = sum(1 for r in results if r.passed)
    for r in results:
        status = "PASS" if r.passed else "FAIL"
        print(f"[{status}] {r.mode.value}: {r.message} ({r.duration_ms:.0f}ms)")

    print(f"\nResults: {passed}/{len(results)} passed")

    if passed < len(results):
        print("\nFailing tests indicate unhandled failure modes in LLMServiceUnderTest.")
        print("Fix the service handler and re-run before deploying.")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Why is code review insufficient validation for the circuit breaker?**

- A. Code review is sufficient. If the logic is correct, it will work.
- B. Tracing through code is not the same as executing the code path. Exception handling, backoff timing, and state transitions have subtle bugs that only appear when running: import ordering issues, wrong exception class caught, state not reset after recovery. The circuit breaker must be triggered in a test before it is trusted.
- C. The circuit breaker should be removed. 529 errors should be retried indefinitely.
- D. Circuit breakers only need to be tested once before the initial deploy.

**2. What is the correct fix and why does this failure mode matter?**

- A. The fix is to validate that the Anthropic SDK never returns invalid JSON. This failure mode does not occur in practice.
- B. Add a try/except around response parsing that catches json.JSONDecodeError and ValueError, returns a structured error response, and logs the raw response body for debugging. This matters because any network interruption mid-stream can produce a truncated JSON body that the SDK cannot parse.
- C. Retry the request immediately on json.JSONDecodeError since it is likely transient.
- D. Switch to a different HTTP client library that handles malformed JSON gracefully.

**3. Is this test correctly verifying the service behavior?**

- A. No. The test should verify that the service retries and eventually gets a valid response, not that it fails after retries.
- B. Yes. The test correctly verifies that the service (a) detects the empty response condition, (b) retries at least once, and (c) returns a structured error when retries are exhausted. The assertion does not need to verify success since the chaos proxy is injecting 100% failure.
- C. No. The test should use failure_rate=0.5 so the second attempt succeeds.
- D. No. Empty responses should not be retried at all.

**4. What should the correct service behavior be for this scenario?**

- A. Pass the truncated response to the user. stop_reason='max_tokens' is a valid API response.
- B. Detect stop_reason='max_tokens', either (a) increase max_tokens and retry if the response is expected to be longer, or (b) add an explicit '[response truncated due to length limit]' indicator to the user, or (c) return an error. Never silently pass truncated content as a complete response.
- C. Retry the request with a lower max_tokens setting.
- D. Log the truncation and pass the response through. Truncation is expected and users understand it.

**5. What is the engineering tradeoff and correct recommendation?**

- A. Yes. 5 seconds is better. Faster recovery always reduces user impact.
- B. The shorter the open period, the more quickly your circuit breaker sends probe requests to an API that may still be overloaded. If the API is having a 5-minute partial outage and your circuit resets every 5 seconds, you send a probe every 5 seconds - 60 probes during the outage. With 60-second resets, you send 5 probes. The right tradeoff depends on your user tolerance for fallback responses vs. the risk of hammering an already-stressed API.
- C. Never reduce the open period below 30 seconds.
- D. The circuit breaker open period should be configurable by the end user.

**6. What do you do during and after the incident?**

- A. During: add a streaming timeout. After: close the incident. No new test needed because it was a one-time event.
- B. During: add a per-chunk streaming timeout (e.g., 5 seconds between chunks) so slow-streaming requests fail fast instead of waiting 30s. After: add FailureMode.SLOW_STREAMING to the chaos suite that simulates 500ms delays between chunks, and verify the per-chunk timeout triggers correctly.
- C. During: increase the timeout to 60 seconds. After: add monitoring for requests taking > 30 seconds.
- D. During: route traffic to OpenAI as a backup. After: add provider failover logic.

**7. What is the correct testing strategy?**

- A. The senior engineer is right. Only real-API tests catch real failure modes.
- B. ChaosProxy tests verify that your service's error handling code paths execute correctly: the right exceptions are caught, retries happen, structured errors are returned. Real-API chaos tests are not practical because you cannot force the real API to return a 529 on demand. Use ChaosProxy for systematic failure mode coverage; use real-API tests for integration smoke tests. The two are complementary.
- C. Only use real-API tests. Mock proxies give false confidence.
- D. Both are equivalent. Choose based on test speed.

_Answers are in `checks.json` in the lesson directory._